# RAPID Alert Generation — Background

This notebook explains **how** the alert dataset is produced. It documents
the pipeline architecture, calibration chain, and data provenance.

## Pipeline Overview

**RAPID** (Roman Alerts Promptly from Image Differencing) processes Roman WFI
exposures through a difference-imaging pipeline:

```
Science image (DN/s)         Template image (DN/s, gain-matched coadd)
         \                        /
          \                      /
           ZOGY subtraction
                  |
         Difference image (DN/s)
                  |
      ┌───────────┼────────────┐
      │           │            │
  SExtractor   PSF fitting   Inject catalog
  (detection)  (quality)     (truth sources)
      │           │            │
      └───────────┼────────────┘
                  |
           Alert generation
                  |
         ┌────────┼────────┐
         │        │        │
    Avro alerts  Truth   Provenance
                sidecar  (headers, catalogs)
```

The data comes from the **Open Universe 2024** (OU2024) Roman Time Domain Survey
simulation. Each job processes one SCA (Sensor Chip Assembly) detector from one visit.

In [ ]:
import glob
import io
import json
import math
import os

import fastavro
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.io import fits

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

DATA_ROOT = '../data/20260227'
EXAMPLE_JID = 'jid1061'

## 1. FITS Headers — Exposure Metadata

Each job's difference image FITS header contains the key metadata:
EXPID, SCA_NUM, FIELD, MJD-OBS, FILTER, ZPTMAG, EXPTIME, BUNIT.

These are saved as JSON in `provenance/headers/` for reproducibility.

In [ ]:
# Load a saved header
hdr_path = '../provenance/headers/jid1061_header.json'
with open(hdr_path) as f:
    hdr = json.load(f)

key_fields = ['FILTER', 'SCA_NUM', 'EXPID', 'FIELD', 'MJD-OBS',
              'ZPTMAG', 'EXPTIME', 'BUNIT', 'CRVAL1', 'CRVAL2']
print(f'Header for {EXAMPLE_JID}:')
for k in key_fields:
    print(f'  {k:12s} = {hdr.get(k, "N/A")}')

print(f'\nAll 15 jobs:')
hdr_files = sorted(glob.glob('../provenance/headers/jid*_header.json'))
print(f'{"JID":12s} {"Filter":8s} {"SCA":>4s} {"MJD":>10s} {"ZPTMAG":>8s} {"EXPTIME":>8s}')
for hf in hdr_files:
    with open(hf) as f:
        h = json.load(f)
    jid = os.path.basename(hf).replace('_header.json', '')
    print(f'{jid:12s} {h.get("FILTER","?"):8s} {h.get("SCA_NUM","?"):>4s} '
          f'{h.get("MJD-OBS","?"):>10s} {h.get("ZPTMAG","?"):>8s} {h.get("EXPTIME","?"):>8s}')

## 2. Photometric Calibration

Converting image pixel values to physical fluxes requires three components:

| Component | What it accounts for | Value range |
|-----------|---------------------|-------------|
| **BANDZPT** | Filter bandpass transmission (sensitivity) | 14.6 -- 15.3 |
| **ZPTMAG** | Collecting area + exposure time (from OU headers, for DN) | 16.5 -- 18.8 |
| **EXPTIME** | Exposure duration (images normalized to DN/s) | 102 -- 901 s |

**Images are in DN/s** (counts per second). The effective zero-point for DN/s is:

$$\mathrm{ZP}_{\mathrm{eff}} = \mathrm{BANDZPT} + \mathrm{ZPTMAG} - 2.5 \, \log_{10}(\mathrm{EXPTIME})$$

Then: $m_{\mathrm{AB}} = -2.5 \, \log_{10}(f_{\mathrm{DN/s}}) + \mathrm{ZP}_{\mathrm{eff}}$

And finally to nanojansky: $f_{\mathrm{nJy}} = 3.631 \times 10^{12} \times 10^{-m_{\mathrm{AB}}/2.5}$

In [ ]:
# The three calibration components
BANDZPT = {
    'F184': 14.622, 'F158': 15.074, 'F129': 15.040,
    'F106': 15.024, 'F087': 14.964, 'F062': 15.297, 'F213': 14.579,
}
ZPTMAG = {
    'F184': 18.824, 'F158': 17.638, 'F129': 17.638,
    'F213': 18.824, 'F062': 16.954, 'F106': 17.638, 'F087': 16.455,
}
EXPTIME = {
    'F184': 901.175, 'F158': 302.275, 'F129': 302.275,
    'F213': 901.175, 'F062': 161.0, 'F106': 302.275, 'F087': 101.7,
}

print(f'{"Band":<6s} {"BANDZPT":>8s} {"ZPTMAG":>8s} {"EXPTIME":>8s} {"ZP_eff":>8s}')
print('-' * 46)
for band in sorted(BANDZPT):
    zp_eff = BANDZPT[band] + ZPTMAG[band] - 2.5 * math.log10(EXPTIME[band])
    print(f'{band:<6s} {BANDZPT[band]:8.3f} {ZPTMAG[band]:8.3f} {EXPTIME[band]:8.1f} {zp_eff:8.3f}')

print('\nEffective ZPs are ~26-27 AB, consistent with Roman WFI single-visit depth.')

### Conversion recipes

```python
# Alert psfFlux (already in nJy) → AB magnitude
mag_AB = -2.5 * np.log10(psfFlux_nJy) + 31.4

# Raw image flux (DN/s) → AB magnitude
mag_AB = -2.5 * np.log10(flux_dns) + ZP_eff[band]

# Truth catalog 'mag' column → AB magnitude (BANDZPT already applied)
mag_AB = truth_mag + ZPTMAG[band]

# Inject catalog flux (DN) → AB magnitude
mag_AB = -2.5 * np.log10(flux_DN) + BANDZPT[band] + ZPTMAG[band]
```

## 3. SExtractor Catalog — Source Detection

Each job runs SExtractor on the difference image with ZP=0, producing a
catalog (`diffimage_masked.txt`) with columns like:
NUMBER, X, Y, RA, Dec, MAG_BEST, MAG_AUTO, FLUX_BEST, FLAGS, CLASS_STAR, etc.

The alert generator parses these comment-delimited catalogs.

In [ ]:
# Parse a SExtractor catalog (same logic as generate_alerts_v100.py)
sex_path = os.path.join(DATA_ROOT, EXAMPLE_JID, 'diffimage_masked.txt')

col_map = {}
rows = []
with open(sex_path) as f:
    for line in f:
        line = line.rstrip('\n')
        if line.startswith('#'):
            parts = line.split()
            if len(parts) >= 3:
                try:
                    col_map[int(parts[1])] = parts[2]
                except ValueError:
                    pass
        elif line.strip():
            vals = line.split()
            row = {}
            for col_num, name in col_map.items():
                idx = col_num - 1
                if idx < len(vals):
                    try:
                        row[name] = float(vals[idx])
                    except ValueError:
                        row[name] = None
            rows.append(row)

sex_df = pd.DataFrame(rows)
print(f'SExtractor catalog for {EXAMPLE_JID}: {len(sex_df)} detections')
print(f'Columns: {list(sex_df.columns)}')
print(f'\nFirst 5 rows (key columns):')
key_cols = [c for c in ['NUMBER', 'ALPHAWIN_J2000', 'DELTAWIN_J2000',
            'XWIN_IMAGE', 'YWIN_IMAGE', 'MAG_BEST', 'MAGERR_BEST',
            'FLUX_BEST', 'FLAGS', 'CLASS_STAR'] if c in sex_df.columns]
sex_df[key_cols].head()

## 4. PSF-Fit Quality Catalog

A separate PSF-fitting step produces `diffimage_masked_psfcat.parquet` with
morphological quality metrics: `sharpness`, `roundness1`, `roundness2`, `peak`,
`qfit`, `cfit`, `redchi`, `npixfit`.

These are matched to SExtractor detections by pixel position (within 5 px radius)
and stored in the alert's `diaSource` record. They are the primary features for
real/bogus and star/galaxy classification (Tracks 3 and 4).

In [ ]:
psf_path = os.path.join(DATA_ROOT, EXAMPLE_JID, 'diffimage_masked_psfcat.parquet')
psf_df = pd.read_parquet(psf_path)
print(f'PSF-fit catalog for {EXAMPLE_JID}: {len(psf_df)} sources')
print(f'Columns: {list(psf_df.columns)}')

# Show how matching works
from scipy.spatial import cKDTree
psf_xy = np.column_stack([psf_df['x_fit'].values, psf_df['y_fit'].values])
psf_tree = cKDTree(psf_xy)

# Match first SExtractor source
sx, sy = sex_df.iloc[0]['XWIN_IMAGE'], sex_df.iloc[0]['YWIN_IMAGE']
dist, idx = psf_tree.query([sx, sy], distance_upper_bound=5.0)
if idx < len(psf_df):
    match = psf_df.iloc[idx]
    print(f'\nSExtractor source at ({sx:.1f}, {sy:.1f}) matched to PSF source at '
          f'({match["x_fit"]:.1f}, {match["y_fit"]:.1f}), dist={dist:.2f} px')
    print(f'  sharpness={match["sharpness"]:.3f}  roundness1={match["roundness1"]:.3f}  '
          f'peak={match["peak"]:.3f}  redchi={match["redchi"]:.3f}')

## 5. Open Universe Truth Catalog

The OU2024 simulation provides a truth catalog for each exposure:
`Roman_TDS_index_{FILTER}_{pointing}_{SCA}.txt`

Columns: `object_id, ra, dec, x, y, realized_flux, flux, mag, obj_type`

- `realized_flux` is in DN (total counts, includes noise realization)
- `flux` is the ideal model flux in DN
- `mag` has BANDZPT already applied: `mag = -2.5 * log10(flux) + BANDZPT`
- `obj_type`: `galaxy`, `star`, `transient`

The alert generator cross-matches each SExtractor detection to the nearest
truth source within 3 pixels. Matched sources get an `obj_type` label;
unmatched detections are artifacts (labeled `None` in the truth sidecar).

In [ ]:
truth_files = sorted(glob.glob(os.path.join(DATA_ROOT, EXAMPLE_JID, 'Roman_TDS_index_*.txt')))
truth_path = truth_files[0]
print(f'Truth catalog: {os.path.basename(truth_path)}')

truth = pd.read_csv(truth_path, sep=r'\s+', comment='#',
    names=['object_id', 'ra', 'dec', 'x', 'y', 'realized_flux', 'flux', 'mag', 'obj_type'])
print(f'{len(truth)} truth sources')
print(f'\nobj_type distribution:')
print(truth['obj_type'].value_counts())

# Show truth → AB magnitude conversion
band = 'F184'  # jid1061 is F184
truth['mag_AB'] = truth['mag'] + ZPTMAG[band]
print(f'\nTruth AB magnitude range (via mag + ZPTMAG[{band}]):')
for otype in ['galaxy', 'star', 'transient']:
    sub = truth.loc[truth['obj_type'] == otype, 'mag_AB']
    print(f'  {otype:12s}: {sub.min():.1f} -- {sub.max():.1f} AB  (N={len(sub)})')

## 6. Cutout Stamp Extraction

Three full-frame FITS images are loaded per job:
- `diffimage_masked.fits` — ZOGY difference image
- `bkg_subbed_science_image.fits` — background-subtracted science image
- `awaicgen_output_mosaic_image_resampled_gainmatched.fits` — gain-matched template

For each SExtractor detection at pixel (x, y), a 129x129 stamp (±64 pixels)
is extracted and serialized as raw FITS bytes in the alert packet.
Sources within 64 pixels of the image edge get no stamps (~6.5% of alerts).

In [ ]:
STAMP_SIZE = 64  # half-width; stamp is 2*64+1 = 129 pixels

# Load the three images for one job
job_dir = os.path.join(DATA_ROOT, EXAMPLE_JID)
diff_path = os.path.join(job_dir, 'diffimage_masked.fits')
sci_path  = os.path.join(job_dir, 'bkg_subbed_science_image.fits')
ref_path  = os.path.join(job_dir, 'awaicgen_output_mosaic_image_resampled_gainmatched.fits')

for label, path in [('Difference', diff_path), ('Science', sci_path), ('Template', ref_path)]:
    if os.path.exists(path):
        with fits.open(path, memmap=True) as hdul:
            data = hdul[0].data
            bunit = hdul[0].header.get('BUNIT', 'N/A')
        print(f'{label:12s}: {data.shape}  BUNIT={bunit}  '
              f'median={np.nanmedian(data):.4f}  std={np.nanstd(data):.4f}')
    else:
        print(f'{label:12s}: not found (download from S3 first)')

# Demonstrate stamp extraction
if os.path.exists(diff_path):
    with fits.open(diff_path, memmap=True) as hdul:
        diff_data = hdul[0].data
    xpos = int(sex_df.iloc[10]['XWIN_IMAGE'])
    ypos = int(sex_df.iloc[10]['YWIN_IMAGE'])
    stamp = diff_data[ypos-STAMP_SIZE:ypos+STAMP_SIZE+1,
                      xpos-STAMP_SIZE:xpos+STAMP_SIZE+1]
    print(f'\nStamp at ({xpos}, {ypos}): shape={stamp.shape}')

    fig, ax = plt.subplots(figsize=(4, 4))
    vmin, vmax = np.nanpercentile(stamp, [5, 95])
    ax.imshow(stamp, origin='lower', cmap='gray', vmin=vmin, vmax=vmax)
    ax.plot(STAMP_SIZE, STAMP_SIZE, 'c+', ms=10, mew=1)
    ax.set_title(f'Diff stamp at ({xpos}, {ypos})')
    plt.tight_layout()
    plt.show()

## 7. Light Curve Tiles (HATS Catalog)

Historical detections of each sky object are stored in a HEALPix-partitioned
parquet catalog (HATS format) on S3. The alert generator:

1. Determines which HEALPix tile covers the job's sky position (tries Norder 4, 5, 6)
2. Downloads the tile parquet file (~50-65 MB each)
3. Builds a 3D unit-vector cKDTree on the tile's (RA, Dec) positions
4. For each SExtractor detection, finds the nearest catalog match within 1 arcsec
5. Extracts the full multi-epoch light curve from the matched row's `nested_lc_data`

The light curve data includes per-epoch: `expid`, `mjdobs`, `fid` (filter ID),
`fluxfit` (DN/s), `fluxerr`, `sca`, `sid` (source ID).

This becomes `prvDiaSources` in the alert — but **only epochs before the current
detection** are included (no future data leakage).

In [ ]:
# Show a light curve tile structure
lc_tiles = sorted(glob.glob('../data/lc_tiles/*.parquet'))
if lc_tiles:
    tile_path = lc_tiles[0]
    lc_df = pd.read_parquet(tile_path)
    print(f'LC tile: {os.path.basename(tile_path)}')
    print(f'  {len(lc_df):,} objects')
    print(f'  Columns: {list(lc_df.columns)}')

    # Show one object's nested light curve data
    row = lc_df.iloc[0]
    nd = row['nested_lc_data']
    print(f'\n  Example object (aid={row["aid"]}):')
    print(f'    meanra={row["meanra"]:.5f}  meandec={row["meandec"]:.5f}  nsources={row["nsources"]}')
    print(f'    nested_lc_data keys: {list(nd.keys())}')
    print(f'    N epochs: {len(nd["mjdobs"])}')

    # Filter ID → band mapping
    FID_TO_FILTER = {1: 'F184', 2: 'F158', 3: 'F129', 4: 'F213',
                     5: 'F062', 6: 'F106', 7: 'F087'}
    from collections import Counter
    fid_counts = Counter(int(f) for f in nd['fid'])
    print(f'    Epochs per filter: {dict((FID_TO_FILTER.get(k,k), v) for k,v in fid_counts.items())}')
else:
    print('No LC tiles found in ../data/lc_tiles/ — download from S3 to explore.')

## 8. Avro Schema

Alerts are serialized in [Apache Avro](https://avro.apache.org/) format using
a 6-file schema under the `rapid.v01_00` namespace:

| File | Record | Fields |
|------|--------|--------|
| `rapid.v01_00.diaSource.avsc` | Detection record | 63 fields (position, flux, shape, flags) |
| `rapid.v01_00.diaObject.avsc` | Persistent object | 49 fields (per-filter flux stats) |
| `rapid.v01_00.alert.avsc` | Top-level wrapper | 11 fields (source, object, history, cutouts) |
| `rapid.v01_00.diaForcedSource.avsc` | Forced photometry | Stub (not populated) |
| `rapid.v01_00.ssSource.avsc` | Solar system | Stub (not populated) |
| `rapid.v01_00.mpc_orbits.avsc` | MPC orbits | Stub (not populated) |

The schema is LSST-compatible by design — field names match Rubin Observatory conventions
(`diaSource`, `diaObject`, `prvDiaSources`, `psfFlux`, `band`, `midpointMjdTai`).

In [ ]:
# Show the top-level alert schema
schema_dir = '../schema/01/00'
alert_schema_path = os.path.join(schema_dir, 'rapid.v01_00.alert.avsc')
if os.path.exists(alert_schema_path):
    with open(alert_schema_path) as f:
        alert_schema = json.load(f)
    print(f'Schema: {alert_schema["namespace"]}.{alert_schema["name"]} v{alert_schema["version"]}')
    print(f'\nTop-level fields:')
    for field in alert_schema['fields']:
        ftype = field['type']
        if isinstance(ftype, list):
            ftype = ' | '.join(str(t) if isinstance(t, str) else t.get('type', str(t)) for t in ftype)
        elif isinstance(ftype, dict):
            ftype = ftype.get('type', str(ftype))
        print(f'  {field["name"]:25s} {str(ftype):30s} {field.get("doc", "")}')
else:
    print(f'Schema not found at {schema_dir}')

## 9. Alert Assembly — Putting It All Together

For each SExtractor detection, the alert generator:

1. **Reads SExtractor row** → position, raw flux, flags, CLASS_STAR
2. **Matches PSF-fit catalog** (within 5 px) → sharpness, roundness, peak, qfit, redchi
3. **Computes calibrated flux**: `mag_AB = MAG_SE + ZP_eff`, then `nJy = 3.631e12 * 10^(-mag/2.5)`
4. **Matches truth catalog** (within 3 px) → obj_type, truth_mag (written to sidecar, not alert)
5. **Matches light curve tile** (within 1 arcsec on sky) → builds `prvDiaSources` and `diaObject`
6. **Extracts stamps** (±64 px) from difference, science, and template images
7. **Computes HEALPix** indices (hp6, hp9) from RA/Dec
8. **Serializes to Avro** and writes to the output file

Key temporal rule for `prvDiaSources`: only epochs with `mjdobs < current_mjd`
and `expid != current_expid` are included — no future data leakage.

In [ ]:
# Verify the generated alerts: spot-check one
with open(f'../output/{EXAMPLE_JID}_alerts.avro', 'rb') as f:
    reader = fastavro.reader(f)
    schema = reader.writer_schema
    alerts = list(reader)

print(f'{len(alerts)} alerts from {EXAMPLE_JID}')

# Find an alert with all components populated
rich = next((a for a in alerts
             if a['prvDiaSources'] and a['diaObject']
             and a['cutoutDifference']), None)
if rich:
    src = rich['diaSource']
    obj = rich['diaObject']
    prv = rich['prvDiaSources']
    flux = src['psfFlux']
    mag = -2.5 * math.log10(flux) + 31.4 if flux and flux > 0 else None

    print(f'\nExample fully-populated alert:')
    print(f'  diaSourceId:   {rich["diaSourceId"]}')
    print(f'  band:          {src["band"]}')
    print(f'  psfFlux:       {flux:.1f} nJy  ({mag:.1f} AB)' if mag else f'  psfFlux: None')
    print(f'  SNR:           {src["snr"]:.1f}')
    print(f'  sharpness:     {src["sharpness"]}')
    print(f'  diaObject:     {obj["nDiaSources"]} historical detections')
    print(f'  prvDiaSources: {len(prv)} prior epochs')
    print(f'  cutouts:       diff={rich["cutoutDifference"] is not None}  '
          f'sci={rich["cutoutScience"] is not None}  '
          f'tpl={rich["cutoutTemplate"] is not None}')

## 10. Dataset Summary

| Property | Value |
|----------|-------|
| Pipeline run | 20260227 |
| Simulation | Open Universe 2024 Roman TDS |
| Jobs | 15 (5 F184 + 10 F158) |
| Total alerts | 38,241 |
| Real (truth-matched) | 13,884 (36.3%) |
| Bogus (artifacts) | 24,357 (63.7%) |
| With light curves | ~57% |
| With cutout stamps | ~93.4% |
| Stamp size | 129 x 129 px (0.11 arcsec/px) |
| MJD range | 62022 -- 62726 (~2 years) |
| Schema | rapid.v01_00 (LSST-compatible) |
| Reference images | Gain-matched coadd templates |
| Magnitude range (diff) | ~17 -- 31 AB |